### Dim Date

In [8]:
from pyspark.sql.functions import col, expr, dayofweek, weekofyear, month, year, quarter, date_format, dayofmonth, monotonically_increasing_id
from pyspark.sql.types import DateType
import datetime

StatementMeta(, fd1cffbc-f114-428d-aad1-e6f435589b81, 64, Finished, Available, Finished, False)

In [9]:
# Generate date ranges
start_date = datetime.date(2020,1,1)
end_date = datetime.date(2024, 12, 31)

# Create a list of dates
date_list = [(start_date + datetime.timedelta(days=x)) for x in range(0, (end_date - start_date).days + 1)]

# Create DataFrame from date list
df_dates = spark.createDataFrame(date_list, DateType()).toDF("date")

# Generate dim_date Dataframe
dim_date = (
    df_dates
        .withColumn("date_key", date_format(col("date"), "yyyy-MM-dd"))
        .withColumn("year", year(col("date")))
        .withColumn("quarter", quarter(col("date")))
        .withColumn("month", month(col("date")))
        .withColumn("day_of_month", dayofmonth(col("date")))
        .withColumn("day_of_week", dayofweek(col("date")))
        .withColumn("week_of_year", weekofyear(col("date")))
        .withColumn("day_name", date_format(col("date"), "EEEE"))
)

dim_date.show()

StatementMeta(, fd1cffbc-f114-428d-aad1-e6f435589b81, 65, Finished, Available, Finished, False)

+----------+----------+----+-------+-----+------------+-----------+------------+---------+
|      date|  date_key|year|quarter|month|day_of_month|day_of_week|week_of_year| day_name|
+----------+----------+----+-------+-----+------------+-----------+------------+---------+
|2020-01-01|2020-01-01|2020|      1|    1|           1|          4|           1|Wednesday|
|2020-01-02|2020-01-02|2020|      1|    1|           2|          5|           1| Thursday|
|2020-01-03|2020-01-03|2020|      1|    1|           3|          6|           1|   Friday|
|2020-01-04|2020-01-04|2020|      1|    1|           4|          7|           1| Saturday|
|2020-01-05|2020-01-05|2020|      1|    1|           5|          1|           1|   Sunday|
|2020-01-06|2020-01-06|2020|      1|    1|           6|          2|           2|   Monday|
|2020-01-07|2020-01-07|2020|      1|    1|           7|          3|           2|  Tuesday|
|2020-01-08|2020-01-08|2020|      1|    1|           8|          4|           2|Wednesday|

In [10]:
# Store dim date as a delta table
dim_date.write.format("delta").mode("overwrite").save("Tables/dbo/dim_date")

StatementMeta(, fd1cffbc-f114-428d-aad1-e6f435589b81, 66, Finished, Available, Finished, False)

### Dim Location

In [11]:
# Load data
df = spark.read.format("csv").option("header", "true").load("Files/COVID_19.csv")

# select distint contries and region
dim_location = (
    df.withColumn("iso_country", col("geoId"))
        .select("countriesAndTerritories", "continentExp", "iso_country")
        .distinct()
        .withColumn("location_key", monotonically_increasing_id())
        .withColumn("effective_start_date", expr("date('2020-01-01')"))
        .withColumn("effective_end_date", expr("date('2999-12-31')"))
        .withColumn("current_flag", expr("TRUE"))
)

dim_location.show()

StatementMeta(, fd1cffbc-f114-428d-aad1-e6f435589b81, 67, Finished, Available, Finished, False)

+-----------------------+------------+-----------+------------+--------------------+------------------+------------+
|countriesAndTerritories|continentExp|iso_country|location_key|effective_start_date|effective_end_date|current_flag|
+-----------------------+------------+-----------+------------+--------------------+------------------+------------+
|                 Canada|     America|         CA|           0|          2020-01-01|        2999-12-31|        true|
|                 Cyprus|      Europe|         CY|           1|          2020-01-01|        2999-12-31|        true|
|                Germany|      Europe|         DE|           2|          2020-01-01|        2999-12-31|        true|
|            South_Sudan|      Africa|         SS|           3|          2020-01-01|        2999-12-31|        true|
|             Kazakhstan|        Asia|         KZ|           4|          2020-01-01|        2999-12-31|        true|
|               Anguilla|     America|         AI|           5| 

In [12]:
dim_location.write.format("delta").mode("overwrite").save("Tables/dbo/dim_location")

StatementMeta(, fd1cffbc-f114-428d-aad1-e6f435589b81, 69, Finished, Available, Finished, False)